# 03 — Feature Extraction: Functional Connectivity (Power-264)

Extracts pairwise Pearson correlations between the **264 ROIs** defined by Power et al. (2011)  
from DPARSF-preprocessed ROISignals `.mat` files.

**Input:** `data/raw/ROISignals_FunImgARglobalCWF/*.mat`  
**Output:** `data/processed/connectivity_power264.csv` — shape `(n_subjects, 34717)`

---
### Power-264 column mapping inside ROISignals (DPARSF v4.3+)
| Block | Columns (1-indexed) | Python slice |
|---|---|---|
| AAL 116 ROIs | 1 – 116 | `[:, 0:116]` |
| Harvard-Oxford cortical | 117 – 212 | `[:, 116:212]` |
| ... | ... | ... |
| Power-264 ROIs | **1570 – 1833** | **`[:, 1569:1833]`** |
| Schaefer-400 | 1834 – 2233 | `[:, 1833:2233]` |

In [ ]:
import os
import re
import glob
import logging
import numpy as np
import pandas as pd
from scipy.io import loadmat

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger(__name__)

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
ROI_DIR    = "../data/raw/ROISignals_FunImgARglobalCWF"
OUTPUT_CSV = "../data/processed/connectivity_power264.csv"

# ── Power-264 slice (0-indexed) ────────────────────────────────────────────
POWER_START    = 1569
POWER_END      = 1833
N_POWER        = POWER_END - POWER_START   # 264
N_FEATURES     = N_POWER * (N_POWER - 1) // 2  # 34,716
MIN_TIMEPOINTS = 30

print(f"Power-264 ROIs : {N_POWER}")
print(f"FC features    : {N_FEATURES:,}")

In [ ]:
def load_roi_signals(mat_path: str) -> np.ndarray:
    """Load and validate ROISignals from a DPARSF .mat file."""
    data = loadmat(mat_path)

    if "ROISignals" not in data:
        raise KeyError(f"'ROISignals' key not found in {mat_path}")

    roi = data["ROISignals"]

    if roi.ndim != 2:
        raise ValueError(f"ROISignals is not 2D (shape {roi.shape})")
    if roi.shape[1] < POWER_END:
        raise ValueError(f"Only {roi.shape[1]} columns — need ≥ {POWER_END}")
    if roi.shape[0] < MIN_TIMEPOINTS:
        raise ValueError(f"Only {roi.shape[0]} timepoints — need ≥ {MIN_TIMEPOINTS}")

    return roi


def compute_fc_vector(roi_signals: np.ndarray) -> np.ndarray:
    """Return upper-triangle FC vector (34,716,) for Power-264 ROIs."""
    power = roi_signals[:, POWER_START:POWER_END]  # (T, 264)
    corr  = np.corrcoef(power.T)                   # (264, 264)

    if np.isnan(corr).any():
        n_nan = int(np.isnan(corr).sum())
        logger.warning("Replaced %d NaN values (zero-variance ROI)", n_nan)
        corr = np.nan_to_num(corr, nan=0.0)

    tri = np.triu_indices(N_POWER, k=1)
    return corr[tri]


def subject_id_from_path(mat_path: str) -> str:
    fname = os.path.basename(mat_path)
    m = re.search(r"(S\d+-\d+-\d{4})", fname)
    return m.group(1) if m else os.path.splitext(fname)[0]

In [ ]:
mat_files = sorted(glob.glob(os.path.join(ROI_DIR, "*.mat")))
logger.info("Found %d .mat files", len(mat_files))

rows, subject_ids, errors = [], [], []

for i, fp in enumerate(mat_files, start=1):
    try:
        roi = load_roi_signals(fp)
        vec = compute_fc_vector(roi)
        rows.append(vec)
        subject_ids.append(subject_id_from_path(fp))
    except Exception as exc:
        errors.append({"file": os.path.basename(fp), "reason": str(exc)})
        logger.warning("Skipped %s — %s", os.path.basename(fp), exc)

    if i == 1 or i % 200 == 0 or i == len(mat_files):
        logger.info("  Processed %d / %d", i, len(mat_files))

print(f"\nValid subjects : {len(subject_ids)}")
print(f"Failed files   : {len(errors)}")

In [ ]:
if errors:
    errors_df = pd.DataFrame(errors)
    print(errors_df.to_string(index=False))
else:
    print("No errors found.")

In [ ]:
feature_cols = [f"fc_{i}" for i in range(N_FEATURES)]

connectivity_df = pd.DataFrame(np.vstack(rows), columns=feature_cols)
connectivity_df.insert(0, "subject_id", subject_ids)

print("DataFrame shape:", connectivity_df.shape)

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
connectivity_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved to: {OUTPUT_CSV}")

### Quick sanity check

In [ ]:
df_check = pd.read_csv(OUTPUT_CSV, nrows=5)
print(df_check.iloc[:, :6])   # subject_id + first 5 FC features
print("\nValue range:", connectivity_df.iloc[:, 1:].min().min().round(4),
      "to", connectivity_df.iloc[:, 1:].max().max().round(4))